In [0]:
df_customers = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/shopkart/bronze/raw_files/olist/olist_customers_dataset.csv")
)


In [0]:
 df_customers.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.customers")

In [0]:
df_customers.show(2)

In [0]:
df_customers.count()

In [0]:
df_customers.schema

# for orders in which we define schema 

In [0]:
df_orders = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/shopkart/bronze/raw_files/olist/olist_orders_dataset.csv")
)

In [0]:
df_orders.schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType
from pyspark.sql.functions import col

schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])


df_orders = spark.read.format("csv").option("header", True).schema(schema).load("/Volumes/shopkart/bronze/raw_files/olist/olist_orders_dataset.csv")

In [0]:
 df_orders.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.orders")

In [0]:

df_orders.count()

# order_items

In [0]:
df_order_items = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/shopkart/bronze/raw_files/olist/olist_order_items_dataset.csv")
)

In [0]:
 df_order_items.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.order_items")

In [0]:
df_order_items.count()

# products

In [0]:
df_products = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/shopkart/bronze/raw_files/olist/olist_products_dataset.csv")
)

In [0]:
 df_products.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.products")

In [0]:
df_products.count()

# payments

In [0]:
df_order_payments = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/shopkart/bronze/raw_files/olist/olist_order_payments_dataset.csv")
)

In [0]:
 df_order_payments.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.payments")

In [0]:
df1 = spark.read.table("shopkart.bronze.customers")
df1.count()

In [0]:
df2 = spark.read.table("shopkart.bronze.orders")
df2.count()


In [0]:
df3 = spark.read.table("shopkart.bronze.order_items")
df3.count()

In [0]:
df4 = spark.read.table("shopkart.bronze.payments")
df4.count()

In [0]:
df5 = spark.read.table("shopkart.bronze.products")
df5.count()

# order_review

In [0]:
df_order_reviews = (spark.read.format("csv")
                   .option("header", True )
                   .option("inferSchema", True)
                   .load("/Volumes/shopkart/bronze/raw_files/olist/olist_order_reviews_dataset.csv")
                   
)

df_order_payments.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.payments")

In [0]:
df_order_reviews.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.order_reviews")

In [0]:
df_sellers = (
            spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .load("/Volumes/shopkart/bronze/raw_files/olist/olist_sellers_dataset.csv")   
)

In [0]:
df_sellers.write.format("delta").mode("overWrite").saveAsTable("shopkart.bronze.sellers")

In [0]:
df_product_category = (spark.read.format("csv")
                       .option("header",True)
                       .option("inferSchema",True)
                       .load("/Volumes/shopkart/bronze/raw_files/olist/product_category_name_translation.csv")
                       
                       )

In [0]:
df_product_category.write.format("delta").mode("overWrite").saveAsTable("shopkart.bronze.product_category")

In [0]:
from pyspark.sql import functions as F

df6 = spark.read \
  .format("csv")\
  .option("header", True)\
  .option("inferSchema", True)\
  .load("/Volumes/shopkart/bronze/raw_files/olist/olist_customers_dataset.csv") \
  .select("*",F.col( "_metadata.file_path").alias("_source_file"))

df6.show(truncate= False)

In [0]:
from pyspark.sql import functions as F

df_customer_time_src_path = df6.withColumn("_ingested_at", F.current_timestamp())
                            
                               


In [0]:
df_customer_time_src_path.show(5)

In [0]:
df_customer_time_src_path.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.customers")

# reuseable function 

In [0]:
import pyspark.sql.functions as F
files = dbutils.fs.ls("/Volumes/shopkart/bronze/raw_files/olist/")

from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType


schema_1 = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

def reuseable(files):
    for file in files:
        if file.name.endswith("csv"):
            if file.name == "olist_orders_dataset.csv":
                df = spark.read.format("csv").option("header",True).schema(schema_1).load(file.path)
                df = df.withColumns({
                            "_ingested_at": F.current_timestamp(),
                                "_source_file":F.lit(file.path)})
                result = file.name.removesuffix('.csv').removesuffix('_dataset').removeprefix('olist_')
                df.write.mode("overwrite").saveAsTable(f"shopkart.bronze.{result}")
            else:
                df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(file.path)
                df = df.withColumns({
                            "_ingested_at": F.current_timestamp(),
                                "_source_file":F.lit(file.path)})
                result = file.name.removesuffix('.csv').removesuffix('_dataset').removeprefix('olist_')
                df.write.mode("overwrite").saveAsTable(f"shopkart.bronze.{result}")
                



reuseable(files)

# just for calculating metadata

In [0]:
from pyspark.sql import functions as F

df6 = spark.read \
  .format("csv")\
  .option("header", True)\
  .option("inferSchema", True)\
  .load("/Volumes/shopkart/bronze/raw_files/olist/olist_customers_dataset.csv") \
  .select("*", "_metadata")

df6.show(truncate= False)

In [0]:
# ---- SK-04: "before" profile of bronze.orders_dirty (run this again on silver.orders for "after") ----
from pyspark.sql import functions as F

df = spark.table("shopkart.bronze.orders")
total = df.count()
profile = {
    "total rows":                total,
    "distinct order_id":         df.select("order_id").distinct().count(),
    "exact duplicate rows":      total - df.distinct().count(),
    "NULL order_status":         df.filter(F.col("order_status").isNull()).count(),
    "non-normalised status":     df.filter(F.col("order_status") != F.lower(F.trim("order_status"))).count(),
    "delivered before purchase": df.filter(F.to_timestamp("order_delivered_customer_date")
                                           < F.to_timestamp("order_purchase_timestamp")).count(),
}
profile["key duplicate rows (total - distinct ids)"] = total - profile["distinct order_id"]
for k, v in profile.items():
    print(f"{k:<45} {v:>7,}")